TODO: Project exploration

# Projects

Ravelry's API exposes an endpoint for `GET /projects/search.json` that returns user-created Project records. Each Project can be linked to one Pattern through `pattern_id`. This will let us combine pattern characteristics with project-level outcomes like completion status, time to finish, and overall engagement.

| **Notable Fields** | **Use Cases** |
| --- | --- |
| `id`, `pattern_id`, `pattern_name` | Identify Projects and join them to Patterns |
| `project_status_id`, `status_name`, `progress` | Measure project outcomes, like % marked completed |
| `started`, `completed` | Estimate time to completion |
| `comments_count`, `favorites_count`, `photos_count` | Analyze user engagement |

### Challenges and Limitations

Unlike other endpoints, Projects cannot be accessed sequentially via ID. To combat this, the `search_pipeline` was run with various queries and parameters. For example,
TODO - SEARCH_SORT_OPTIONS, queries?

However, we cannot know the exact level of sampling. According to Ravelry's documentation, there are at least 33 million Projects.
TODO

### Documentation
- https://www.ravelry.com/api#projects_search
- https://www.ravelry.com/api#Project__result

In [ ]:
con.execute("""
    DESCRIBE raw_projects
""").fetchdf()

In [ ]:
con.execute("""
    SELECT
        SUM(raw_patterns.projects_count) AS total_projects,
        COUNT(raw_projects.*) AS scraped_projects,
        COUNT(raw_projects.pattern_id) AS with_pattern,
        COUNT(DISTINCT raw_projects.pattern_id) AS unique_patterns,
        COUNT(raw_projects.completed) AS completed,
        AVG(
            CAST(raw_projects.completed AS DATE) - CAST(raw_projects.started AS DATE)
        ) AS avg_completion_days
    FROM raw_projects
    JOIN raw_patterns
    ON raw_projects.pattern_id = raw_patterns.id
""").fetchdf()

In [ ]:
con.execute("""
    DESCRIBE processed_projects
""").fetchdf()

In [ ]:
con.execute("""
    SELECT
        COUNT(*) AS total_projects,
        COUNT(pattern_id) AS with_pattern,
        COUNT(DISTINCT pattern_id) AS unique_patterns,
        COUNT(completed) AS completed,
        AVG(
            CAST(completed AS DATE) - CAST(started AS DATE)
        ) AS avg_completion_days
    FROM processed_projects
""").fetchdf()

In [ ]:
# Most Popular Project Patterns
con.execute("""
    SELECT
        p.pattern_id,
        r.name AS pattern_name,
        COUNT(p.*) AS num_projects
    FROM clean_projects AS p
    JOIN raw_patterns AS r
        ON p.pattern_id = r.id
    GROUP BY p.pattern_id, r.name
    HAVING COUNT(p.*) > 250
    ORDER BY num_projects DESC
""").fetchdf()

In [ ]:
con.sql("""
    SELECT
        name
    FROM raw_patterns
    ORDER BY projects_count DESC
    LIMIT 25
""").df()['name'].tolist()

In [ ]:
# TODO - why is 2025 & 2024 missing?

# Most Popular Project Pattern by Year
con.execute("""
    WITH yearly_counts AS (
        SELECT
            pattern_id,
            COUNT(*) AS num_projects,
            EXTRACT(YEAR FROM created_at) AS year
        FROM processed_projects
        GROUP BY year, pattern_id
        ORDER BY year DESC
    ),
    ranked_counts AS (
        SELECT
            pattern_id,
            num_projects,
            year,
            RANK() OVER (PARTITION BY year ORDER BY num_projects DESC) AS rank
        FROM yearly_counts
    )
    SELECT 
        ranks.year,
        patterns.id,
        patterns.name,
        ranks.num_projects
    FROM ranked_counts AS ranks
    JOIN raw_patterns AS patterns
        ON ranks.pattern_id = patterns.id
    WHERE rank = 1
    ORDER BY year DESC
""").fetchdf()

In [ ]:
# Average Completion Time Variation by Year
con.execute("""
    WITH yearly_stats AS (
        SELECT
            EXTRACT(YEAR FROM created_at) AS year,
            ROUND(
                AVG(
                    CAST(completed AS DATE) - CAST(started AS DATE)
                )
                , 2
            ) AS avg_completion_time
        FROM processed_projects
        WHERE (
            completed IS NOT NULL AND 
            started IS NOT NULL AND
            completed >= started
        )
        GROUP BY year
        ORDER BY year
    )
    SELECT
        year,
        avg_completion_time,
        ROUND(
            avg_completion_time - LAG(avg_completion_time) OVER (ORDER BY year ASC), 
            2
        ) AS change_in_days,
        ROUND(
            ((avg_completion_time - LAG(avg_completion_time) OVER (ORDER BY year ASC)) 
            / NULLIF(LAG(avg_completion_time) OVER (ORDER BY year ASC), 0)) * 100, 
            2
        ) AS change_in_percentage
    FROM yearly_stats
    ORDER BY year
""").fetchdf()

In [ ]:
# TODO - correlation matrix

SQL = """
    SELECT
        ln(1.0 + favorites_count) AS log_favorites,
        ln(1.0 + comments_count) AS log_comments,
        ln(1.0 + photos_count) AS log_photos,
        ln(1.0 + rating) AS log_rating,
        favorites_count AS favorites,
        comments_count AS comments,
        photos_count AS photos,
        rating
    FROM processed_projects
"""

df = con.execute(SQL).fetchdf()

corr_matrix = df.corr()

cmap = sns.blend_palette([
    "#e7f0f7",
    "#5a0e21"
], as_cmap=True)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap=cmap, vmin=0, vmax=1, ax=ax)

plt.show()